# Step 8 — Airflow schedule and timezone test

This notebook validates the scheduling configuration in both NammaMart DAG files without importing Airflow. It checks the cron expression, timezone-aware start date, catchup policy, retries, retry delay and execution timeout.

The notebook also calculates the equivalent UTC time and documents what happens when the local Docker environment is offline.

In [1]:
from ast import parse, Call, Constant, Dict, keyword
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

# Read source text instead of importing Airflow, so this test can run in
# the local notebook environment as well as inside the Astro container.
DAG_FILES = {
    'ETL': Path('dags/nammamart_etl_samy.py'),
    'ELT': Path('dags/nammamart_elt_samy.py'),
}
EXPECTED_SCHEDULE = '0 9 * * 1-6'

def find_dag_call(tree):
    # Locate the @dag(...) call in the Python syntax tree.
    for node in ast_walk(tree):
        if isinstance(node, Call) and getattr(node.func, 'id', None) == 'dag':
            return node
    raise ValueError('No @dag call found')

def ast_walk(node):
    # A small recursive walker keeps this notebook independent of extra
    # AST libraries and makes the source-level checks easy to understand.
    yield node
    for child in __import__('ast').iter_child_nodes(node):
        yield from ast_walk(child)

def literal_keyword(dag_call, name):
    for item in dag_call.keywords:
        if item.arg == name and isinstance(item.value, Constant):
            return item.value.value
    return None

def dict_keyword(dag_call, name):
    for item in dag_call.keywords:
        if item.arg == name and isinstance(item.value, Dict):
            result = {}
            for key, value in zip(item.value.keys, item.value.values):
                if isinstance(key, Constant) and isinstance(value, Constant):
                    result[key.value] = value.value
            return result
    return {}

for label, path in DAG_FILES.items():
    source = path.read_text()
    tree = parse(source)
    dag_call = find_dag_call(tree)
    values = {
        'dag_id': literal_keyword(dag_call, 'dag_id'),
        'schedule': literal_keyword(dag_call, 'schedule'),
        'catchup': literal_keyword(dag_call, 'catchup'),
        'default_args': dict_keyword(dag_call, 'default_args'),
    }
    # These source-level assertions prevent accidental schedule drift.
    assert values['schedule'] == EXPECTED_SCHEDULE
    assert values['catchup'] is False
    assert values['default_args']['retries'] == 2
    assert '"retry_delay": timedelta(minutes=5)' in source
    assert '"execution_timeout": timedelta(minutes=30)' in source
    print(f'{label}: {values}')

ETL: {'dag_id': 'nammamart_etl_samy', 'schedule': '0 9 * * 1-6', 'catchup': False, 'default_args': {'owner': 'samy', 'retries': 2}}
ELT: {'dag_id': 'nammamart_elt_samy', 'schedule': '0 9 * * 1-6', 'catchup': False, 'default_args': {'owner': 'samy', 'retries': 2}}


In [2]:
# Convert the business schedule from India time to UTC.
run_ist = datetime(2026, 10, 12, 9, 0, tzinfo=ZoneInfo('Asia/Kolkata'))
run_utc = run_ist.astimezone(timezone.utc)
print(f'09:00 Asia/Kolkata = {run_utc:%H:%M} UTC')
assert run_utc.hour == 3 and run_utc.minute == 30

# Explain the cron fields in plain language for the assignment submission.
cron_fields = EXPECTED_SCHEDULE.split()
print(f'Cron fields: minute={cron_fields[0]}, hour={cron_fields[1]}, days={cron_fields[4]}')
print('Meaning: 09:00 every Monday through Saturday in Asia/Kolkata.')
print('If start_date were incorrectly UTC, 09:00 would mean 14:30 IST.')

09:00 Asia/Kolkata = 03:30 UTC
Cron fields: minute=0, hour=9, days=1-6
Meaning: 09:00 every Monday through Saturday in Asia/Kolkata.
If start_date were incorrectly UTC, 09:00 would mean 14:30 IST.


In [3]:
# Record operational behavior that cannot be proven by a local source check.
operational_notes = {
    'Docker/laptop off at 09:00': 'The local scheduler cannot create the run; with catchup=False, it is not automatically backfilled.',
    'Task failure': 'Airflow retries the task twice after five minutes, then marks it failed.',
    'Long-running task': 'The task is stopped after the 30-minute execution timeout.',
}
for situation, behavior in operational_notes.items():
    print(f'{situation}: {behavior}')

print('All Step 8 schedule checks passed.')

Docker/laptop off at 09:00: The local scheduler cannot create the run; with catchup=False, it is not automatically backfilled.
Task failure: Airflow retries the task twice after five minutes, then marks it failed.
Long-running task: The task is stopped after the 30-minute execution timeout.
All Step 8 schedule checks passed.
